# Mart

In [ ]:
CREATE OR REPLACE TABLE marts.fct_fulfillment AS
WITH orders AS (
    SELECT * FROM staging.orders_clean_view
),
shipments AS (
    SELECT * FROM staging.shipments_clean_view
),
final_calculations AS (
    SELECT 
        o.order_id,
        o.customer_id,
        o.region,
        o.order_cost,
        s.shipping_cost,
        s.carrier,
        s.shipping_date,
        s.delivery_date,
        -- All calculations in one place
        DATEDIFF('day', o.order_date, s.shipping_date) AS lead_time_shipping,
        DATEDIFF('day', s.shipping_date, s.delivery_date) AS lead_time_delivery,
        (o.order_cost - s.shipping_cost) AS net_margin_usd
    FROM orders o
    LEFT JOIN shipments s USING(order_id)
)
SELECT 
    *,
    -- Logic built on top of the calculations above
    CASE
        WHEN delivery_date IS NULL AND lead_time_shipping >= 2 THEN 'LOST_OR_DELAYED'
        WHEN lead_time_delivery <= 1 THEN 'FAST_TRACK'
        ELSE 'STANDARD' 
    END AS order_status
FROM final_calculations;